# @jack/lcoe-calculator

Levelized cost of energy, self-served from Colab.

Published version **v1.0.0** — this notebook fetches that exact snapshot from the
PowerAI Hub, not the upstream repository's latest commit.

Run all cells: **Runtime → Run all** (or ⌘/Ctrl + F9).

The Hub does not run primitives. This notebook runs in *your* Colab session, on
Google's hardware, under your account.

[View on the Hub](https://hub.powerai.ai/jack/lcoe-calculator)


In [ ]:
PRIMITIVE = "@jack/lcoe-calculator"
ARCHIVE = "https://hub-api.powerai.ai/api/artifacts/jack/lcoe-calculator/download-archive/"
WORKDIR = "/content/primitive"

import shutil, urllib.request, zipfile
from pathlib import Path

shutil.rmtree(WORKDIR, ignore_errors=True)          # re-runs start clean
Path(WORKDIR).mkdir(parents=True, exist_ok=True)

archive = Path("/content/primitive.zip")
urllib.request.urlretrieve(ARCHIVE, archive)
with zipfile.ZipFile(archive) as zf:
    zf.extractall(WORKDIR)

files = sorted(p for p in Path(WORKDIR).rglob("*") if p.is_file())
print(f"{PRIMITIVE}: {len(files)} files in {WORKDIR}")
for p in files[:20]:
    print(" ", p.relative_to(WORKDIR))
if len(files) > 20:
    print(f"  … and {len(files) - 20} more")


In [ ]:
import subprocess, sys
from pathlib import Path

req = next(Path(WORKDIR).rglob("requirements.txt"), None)
if req is None:
    print("No requirements.txt. Install what you need with pip, e.g. !pip install pandas")
else:
    print(f"Installing {req.relative_to(WORKDIR)} …")
    done = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
        capture_output=True, text=True,
    )
    print("done" if done.returncode == 0 else f"pip failed:\n{done.stderr[-2000:]}")


In [ ]:
import os
os.chdir(WORKDIR)

readme = next((p for p in sorted(Path(WORKDIR).rglob("README*")) if p.is_file()), None)
print(readme.read_text(errors="replace")[:3000] if readme else "No README in this primitive.")


## Serve this primitive as an endpoint

This archive ships its own `powerai_endpoint.py`, so the cells below run *that* —
not a Hub-written reference implementation — and open a public HTTPS tunnel to it.

Two things to know before you rely on it:

* **It dies with this session.** Colab reclaims idle notebooks after ~90 minutes and
  caps sessions around 12 hours. The Hub stores one address per primitive, so when this
  session ends that address answers nothing until you re-run and paste a new one.
* **Each person who runs this gets a different URL.** One Hub, one address — so this is
  a demo you drive, not something visitors wire up themselves.

The next cell downloads `cloudflared` (Cloudflare's official release), because a Colab
VM has no inbound networking of its own.


In [ ]:
import os, re, subprocess, sys, time, urllib.request

PORT = 8011
MODULE = "powerai_endpoint"
CLOUDFLARED = "/usr/local/bin/cloudflared"

# Re-running this cell has to be safe: a previous run leaves a server and a tunnel
# alive, and the next one would otherwise pass the health probe anyway because the
# *stale* server answers it — a stale process serving stale code, reported as fresh.
for pattern in ("cloudflared tunnel", f"uvicorn {MODULE}:app"):
    subprocess.run(["pkill", "-f", pattern], capture_output=True)
time.sleep(1)

if not os.path.exists(CLOUDFLARED):
    got = subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
        f"cloudflared-linux-amd64 -O {CLOUDFLARED} && chmod +x {CLOUDFLARED}",
        shell=True, capture_output=True, text=True,
    )
    if got.returncode != 0:
        raise RuntimeError(f"could not install cloudflared:\n{got.stderr[-1000:]}")
    print("cloudflared installed")
else:
    print("cloudflared already present")

# --app-dir puts WORKDIR on the import path for this one process, so the archive
# need not be pip-installed (and may not even be a package) to be served.
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", f"{MODULE}:app", "--app-dir", WORKDIR,
     "--host", "127.0.0.1", "--port", str(PORT)],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

for _ in range(90):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=1)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(
        f"{MODULE}.py did not come up as a healthy server on :{PORT} — check its "
        "own errors above; this is the archive's own code, not the Hub's"
    )
print(f"endpoint up on :{PORT}")

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
PUBLIC_URL = ""
deadline = time.time() + 90
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        break
    found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if found:
        PUBLIC_URL = found.group(0)
        break
if not PUBLIC_URL:
    raise RuntimeError("no tunnel URL; re-run this cell")

ENDPOINT_URL = PUBLIC_URL + "/run"
print()
print("Endpoint URL (paste into the Tool tab -> Configure endpoint):")
print("   ", ENDPOINT_URL)


In [ ]:
import json, urllib.error, urllib.request

# Built from the tool_interface declared for @jack/lcoe-calculator. This cannot hand-verify a real
# answer the way the Hub's own reference wrapper does — it checks that the contract
# is honoured, not that any particular answer is correct.
tool_names = ['lcoe']

for name in tool_names:
    envelope = {"powerai": "1", "tool": name, "input": {}}
    request = urllib.request.Request(
        ENDPOINT_URL, data=json.dumps(envelope).encode(),
        headers={"Content-Type": "application/json"},
    )
    try:
        reply = json.load(urllib.request.urlopen(request, timeout=90))
    except urllib.error.HTTPError as err:
        reply = json.load(err)
    assert "ok" in reply, f"{name}: response has no 'ok' field — {reply}"
    if reply["ok"]:
        print(f"{name} -> ok (empty input was accepted)")
    else:
        error = reply.get("error", {})
        assert error.get("code") != "unknown_tool", (
            f"{name} was not recognised — check the spelling against its "
            "tool_interface.name"
        )
        print(f"{name} -> recognised, rejected empty input ({error.get('message')})")

print()
print("endpoint speaks the contract for every declared tool — verifying that its")
print("answers are actually correct is on you, not this check.")


### Register it, then clean up

Paste the URL above into the [Tool tab](https://hub.powerai.ai/jack/lcoe-calculator?tab=tool) → *Configure endpoint*. The
primitive then reports `invocable: true`, appears in `?invocable=true`, and is rendered
as an action in `/api/integrations/action-registry/?as=markdown` for an agent platform
to pull.

**When you finish, clear it** — the catalogue should not advertise an address that died
with this session. Same dialog: *Change endpoint* → **Remove**.

Clear it there rather than with `curl`. That API is owner-authenticated, so a bare
request is rejected, and the credential that would satisfy it does not belong in a
notebook this repo publishes.


## Your turn

`@jack/lcoe-calculator` is unpacked in `/content/primitive` and it is the working directory. Add a
cell and use it.

Nothing here is sent back to the Hub — this session is yours, and it disappears when
you close it. Colab's free tier gives you CPU always and a GPU when one is spare.
